# Week 01: Refactoring as a Design Skill
## The House That Jack Built

> The first version of a program that works is rarely the version you should keep.

Early in COMP 170, you were asked to print each stanza of the cumulative nursery rhyme *The House That Jack Built*:

```text
This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate the malt
That lay in the house that Jack built.

This is the cat
That killed the rat that ate the malt
That lay in the house that Jack built.

This is the dog that worried the cat
That killed the rat that ate the malt
That lay in the house that Jack built.

This is the cow with the crumpled horn
That tossed the dog that worried the cat
That killed the rat that ate the malt
That lay in the house that Jack built.
```

We're going to take a solution to this you already wrote once, and rewrite it five more times. Nothing about the *output* will change. Everything about the *code* will.

---

A simple approach to this problem, and the one most of us would reach for first, is shown below: one function per stanza.

In [1]:
def house():
    print("This is the house that Jack built.")


def malt():
    print("This is the malt that lay in the house that Jack built.")


def rat():
    print("This is the rat that ate the malt")
    print("That lay in the house that Jack built.")


def cat():
    print("This is the cat")
    print("That killed the rat that ate the malt")
    print("That lay in the house that Jack built.")


def dog():
    print("This is the dog that worried the cat")
    print("That killed the rat that ate the malt")
    print("That lay in the house that Jack built.")


def main():
    house()
    print()
    malt()
    print()
    rat()
    print()
    cat()
    print()
    dog()


main()


This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate the malt
That lay in the house that Jack built.

This is the cat
That killed the rat that ate the malt
That lay in the house that Jack built.

This is the dog that worried the cat
That killed the rat that ate the malt
That lay in the house that Jack built.


With a bit more effort, the code can be "improved" to the following: each function builds only the part of the stanza that's new, then hands off to the next one inward.

In [1]:
def house():
    print("That lay in the house that Jack built.")


def malt_tail():
    print("the malt")
    house()


def rat_tail():
    print("the rat that ate the malt")
    house()


def cat_tail():
    print("the cat")
    print("That killed the rat that ate the malt")
    house()


def dog_tail():
    print("the dog that worried the cat")
    print("That killed the rat that ate the malt")
    house()


def house_verse():
    print("This is the house that Jack built.")


def malt_verse():
    print("This is the malt that lay in")
    house()


def rat_verse():
    print("This is the rat that ate")
    malt_tail()


def cat_verse():
    print("This is")
    cat_tail()


def dog_verse():
    print("This is")
    dog_tail()


def main():
    house_verse()
    print()
    malt_verse()
    print()
    rat_verse()
    print()
    cat_verse()
    print()
    dog_verse()


main()


This is the house that Jack built.

This is the malt that lay in
That lay in the house that Jack built.

This is the rat that ate
the malt
That lay in the house that Jack built.

This is
the cat
That killed the rat that ate the malt
That lay in the house that Jack built.

This is
the dog that worried the cat
That killed the rat that ate the malt
That lay in the house that Jack built.


This is just about as good as it gets with the basic skills acquired early in an introductory course. The code can be improved a little more by introducing constants, so that each stanza's text is built once instead of retyped inside every function.

In [1]:
PREFIX = "This is"

HOUSE_TAIL = "the house that Jack built."
MALT_TAIL = "the malt that lay in " + HOUSE_TAIL
RAT_TAIL = "the rat that ate\n" + MALT_TAIL
CAT_TAIL = "the cat \nThat killed " + RAT_TAIL
DOG_TAIL = "the dog that worried " + CAT_TAIL


def house_verse():
    print(PREFIX, HOUSE_TAIL)


def malt_verse():
    print(PREFIX, MALT_TAIL)


def rat_verse():
    print(PREFIX, RAT_TAIL)


def cat_verse():
    print(PREFIX, CAT_TAIL)


def dog_verse():
    print(PREFIX, DOG_TAIL)


def main():
    house_verse()
    print()
    malt_verse()
    print()
    rat_verse()
    print()
    cat_verse()
    print()
    dog_verse()


main()


This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate
the malt that lay in the house that Jack built.

This is the cat 
That killed the rat that ate
the malt that lay in the house that Jack built.

This is the dog that worried the cat 
That killed the rat that ate
the malt that lay in the house that Jack built.


Still, we are writing repetitive code. One way to improve the solution further is with lists. We start with some (really bad) code introducing lists and hard-coded indices into the earlier `print` statements.

In [1]:
# Use lists of strings to deliver the nursery rhyme.

PREFIX = "This is the"
SUFFIX = "house that Jack built."

ITEMS = ["", "malt", "rat", "cat", "dog"]
ACTIONS = ["", "lay", "ate", "killed", "worried"]


def house():
    print(f"{PREFIX} {SUFFIX}")


def malt():
    print(f"{PREFIX} {ITEMS[1]} that {ACTIONS[1]} in the {SUFFIX}")


def rat():
    print(
        f"{PREFIX} {ITEMS[2]} that {ACTIONS[2]} the {ITEMS[1]} that {ACTIONS[1]} in the {SUFFIX}"
    )


def cat():
    print(
        f"{PREFIX} {ITEMS[3]} that {ACTIONS[3]} the {ITEMS[2]} that {ACTIONS[2]} the {ITEMS[1]} that {ACTIONS[1]} in the {SUFFIX}"
    )


def dog():
    print(
        f"{PREFIX} {ITEMS[4]} that {ACTIONS[4]} the {ITEMS[3]} that {ACTIONS[3]} the {ITEMS[2]} that {ACTIONS[2]} the {ITEMS[1]} that {ACTIONS[1]} in the {SUFFIX}"
    )


house()
print()
malt()
print()
rat()
print()
cat()
print()
dog()


This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate the malt that lay in the house that Jack built.

This is the cat that killed the rat that ate the malt that lay in the house that Jack built.

This is the dog that worried the cat that killed the rat that ate the malt that lay in the house that Jack built.


The code above reveals a pattern we can exploit further. Focusing on `cat()` and the method before it, `rat()`, notice that `cat()`'s line is just `rat()`'s line with one more clause wrapped around the front — every stanza is built from the one before it. So instead of rebuilding each string from scratch inside every function, let's build each one once, from the previous one.

In [1]:
PREFIX = "This is the"
SUFFIX = "house that Jack built."

ITEMS = ["", "malt", "rat", "cat", "dog"]
ACTIONS = ["", "lay", "ate", "killed", "worried"]

malt_string = f"{ITEMS[1]} that {ACTIONS[1]} in the {SUFFIX}"
rat_string = f"{ITEMS[2]} that {ACTIONS[2]} the {malt_string}"
cat_string = f"{ITEMS[3]} that {ACTIONS[3]} the {rat_string}"
dog_string = f"{ITEMS[4]} that {ACTIONS[4]} the {cat_string}"


def house():
    print(f"{PREFIX} {SUFFIX}")


def malt():
    print(f"{PREFIX} {malt_string}")


def rat():
    print(f"{PREFIX} {rat_string}")


def cat():
    print(f"{PREFIX} {cat_string}")


def dog():
    print(f"{PREFIX} {dog_string}")


house()
print()
malt()
print()
rat()
print()
cat()
print()
dog()


This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate the malt that lay in the house that Jack built.

This is the cat that killed the rat that ate the malt that lay in the house that Jack built.

This is the dog that worried the cat that killed the rat that ate the malt that lay in the house that Jack built.


Now we can see a broader pattern emerging, and we can exploit it. The various item strings (`malt_string`, `rat_string`, and so on) can be generated with a loop iterating over `ITEMS` and `ACTIONS`. And the five near-identical print functions can be replaced with just two: one for the special first stanza, one for every stanza after it.

In [1]:
PREFIX = "This is the"
SUFFIX = "house that Jack built."
CONNECTOR = "that"

ITEMS = ["", "malt", "rat", "cat", "dog"]
ACTIONS = ["", "lay", "ate", "killed", "worried"]


def first_stanza() -> None:
    """Print the poem's first stanza, which has no preceding clause to
    build on and so cannot be produced by the general loop below."""
    print(f"{PREFIX} {SUFFIX}\n")


def other_stanzas() -> None:
    """Print every stanza after the first, each one built by wrapping
    one more clause around the previous stanza's text."""
    # What comes after "in the" for malt, and gets nested from there on.
    prev = f"{SUFFIX}"

    for i in range(1, len(ITEMS)):
        if i == 1:
            # malt is the only one that says "lay in the ...".
            current = f"{ITEMS[i]} {CONNECTOR} {ACTIONS[i]} in the {prev}"
        else:
            # everything else says "... the <previous phrase>".
            current = f"{ITEMS[i]} {CONNECTOR} {ACTIONS[i]} the {prev}"

        print(f"{PREFIX} {current}\n")
        prev = current


first_stanza()
other_stanzas()


This is the house that Jack built.

This is the malt that lay in the house that Jack built.

This is the rat that ate the malt that lay in the house that Jack built.

This is the cat that killed the rat that ate the malt that lay in the house that Jack built.

This is the dog that worried the cat that killed the rat that ate the malt that lay in the house that Jack built.



Everything in the code above is written using techniques you already had in COMP 170. The most important part is realizing the *cumulative* nature of the nursery rhyme, and finding the best way to reflect that in the code itself — not just in the output.

---

### How much does each version grow with the poem?

Suppose the rhyme had 100 stanzas instead of 5, not 5. How much *code* would each version above need you to write or change?

| Version | New functions | New constants / list entries |
|---|---|---|
| Naive pass | ~100, all hand-written | — |
| Shared structure | ~100, still one-off | — |
| Constants | ~100 functions and ~100 constants | 100 |
| Lists, hard-coded indices | ~100 functions | 100 |
| Build-from-previous | ~100 functions | 100 |
| The loop | **0** | 100 |

Every earlier version needs more code, written by a person, in direct proportion to the size of the poem. The final version needs more *data* — longer lists — but the same two functions handle a 5-stanza poem and a 500-stanza poem equally well.

---

### One thing to add

The rhyme at the top of this notebook has six stanzas: house, malt, rat, cat, dog, and **cow**. Every version above only ever handles five, we stop at `dog`.

Try extending `ITEMS`, `ACTIONS`, and `other_stanzas()` to include the cow stanza — *"This is the cow with the crumpled horn that tossed the dog..."* 